In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # para que encuentre los módulos del repo

In [ ]:
from graph.pipeline import run_pipeline
from graph.state import ComplianceState
import json

In [ ]:
alert_moderate = {
    "alert_id": "TEST-001",
    "customer_id": "CUST-001",
    "alert_type": "transferencia_internacional",
    "description": "Transferencia internacional de alto monto desde cuenta corporativa"
}

result_moderate = run_pipeline(alert_moderate)
print("=== RESULTADO MODERATE RISK ===")
print(f"Risk score: {result_moderate.get('risk_score')}")
print(f"Decision: {result_moderate.get('decision')}")
print(f"Confidence: {result_moderate.get('confidence')}")
print(f"Reasoning steps:")
for step in result_moderate.get('reasoning_steps', []):
    print(f"  - {step}")
print(f"\nFinal report: {result_moderate.get('final_report')}")

In [ ]:
# CUST-002 devuelve is_pep=True según el mock de get_customer_data en tools/bigquery_tools.py
# (hash("CUST-002") % 7 == 0 → is_pep: True)
# Si el pipeline no escala automáticamente por risk_score >= 9,
# el DecisionAgent debería escalar por política PEP.
alert_pep = {
    "alert_id": "TEST-002",
    "customer_id": "CUST-002",
    "alert_type": "transferencia_internacional",
    "description": "Transferencia internacional de alto monto desde cuenta de cliente PEP"
}

result_pep = run_pipeline(alert_pep)
print("=== RESULTADO PEP RISK ===")
print(f"Risk score: {result_pep.get('risk_score')}")
print(f"Es PEP: {result_pep.get('customer', {}).get('is_pep')}")
print(f"Decision: {result_pep.get('decision')}")
print(f"Confidence: {result_pep.get('confidence')}")
print(f"Reasoning steps:")
for step in result_pep.get('reasoning_steps', []):
    print(f"  - {step}")
print(f"\nFinal report: {result_pep.get('final_report')}")

In [ ]:
# Ver el estado completo sin el transaction_history (muy verboso)
state_summary = {k: v for k, v in result_moderate.items() if k != "transaction_history"}
print(json.dumps(state_summary, indent=2, default=str))

In [ ]:
assert result_moderate.get("decision") in ("escalate", "dismiss", "request_info"), "Decision debe estar seteada"
assert result_moderate.get("reasoning_steps"), "Debe haber reasoning steps"
print("✓ Audit trail completo")